# **Get Price Data**

In [ ]:
import requests
import pandas as pd
import datetime as dt
from tabulate import tabulate
import numpy as np
import time
import matplotlib.pyplot as plt
from pathlib import Path

API_KEY = "your-coingecko-pro-key-here"
headers = {
    "accept": "application/json",
    "x-cg-pro-api-key": API_KEY
}

In [ ]:
# ==============================
# STEP 1: Get Top 500 Tokens Today
# ==============================
def get_top_500_coins_table():
    url = "https://pro-api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": 250,
    }

    coins = []
    for page in range(1, 4):  # up to 750
        params["page"] = page
        response = requests.get(url, headers=headers, params=params)
        data = response.json()
        coins.extend(data)

    # Exclusion keywords
    exclude_keywords = [
        "usd", "usdt", "usdc", "busd", "dai", "tusd", "usdp", "gusd", "lusd", "fdusd",
        "usdd", "susd", "eusd", "wrapped", "wbtc", "weth", "renbtc", "staked", "stake"
    ]
    exclude_ids = [
        "bridged-wrapped-ether-starkgate","sbtc-2","wrapped-zenbtc","liquid-hype-yield",
        "compound-ether","binance-peg-sol","bitcoin-avalanche-bridged-btc-b","binance-peg-dogecoin",
        "tbtc","clbtc","tether-gold","rocket-pool-eth","solv-btc","pax-gold",
        "cgeth-hashkey","frax-ether","resolv-usr","jupiter-perpetual","gho", "stasis-eurs", "dola-usd", "blockchain-capital",
        "ousg", "mbg-by-multibank-group", "tradable-na-rent-financing-platform-sstn", "kinesis-gold", "kinesis-silver", "spiko-us-t-bills-money-market-fund",
        "onyc", "tradable-singapore-fintech-ssl-2", "vaneck-treasury-fund"
    ]

    def is_excluded(coin):
        name = coin["name"].lower()
        symbol = coin["symbol"].lower()
        cid = coin["id"].lower()
        return (any(kw in name or kw in symbol for kw in exclude_keywords) or cid in exclude_ids)

    included = [c for c in coins if not is_excluded(c)]

    df_included = pd.DataFrame(included)[["id", "symbol", "name", "current_price", "market_cap"]].head(500)
    df_included["market_cap_millions"] = (df_included["market_cap"] / 1_000_000).round(2)

    return df_included

In [ ]:
# ==============================
# Fetch historical daily prices
# ==============================
def get_historical_prices(coin_id, start_date="2015-01-01", end_date=None):
    """
    Fetch daily historical prices for a token from CoinGecko between start_date and end_date.
    Returns a DataFrame with [date, price].
    """
    if end_date is None:
        end_date = dt.date.today().strftime("%Y-%m-%d")

    # Convert to UNIX timestamps
    start_ts = int(dt.datetime.strptime(start_date, "%Y-%m-%d").timestamp())
    end_ts = int(dt.datetime.strptime(end_date, "%Y-%m-%d").timestamp())

    url = f"https://pro-api.coingecko.com/api/v3/coins/{coin_id}/market_chart/range"
    params = {
        "vs_currency": "usd",
        "from": start_ts,
        "to": end_ts
    }

    r = requests.get(url, headers=headers, params=params)
    if r.status_code != 200:
        return None

    data = r.json()
    if "prices" not in data:
        return None

    df = pd.DataFrame(data["prices"], columns=["timestamp", "price"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms").dt.date
    df = df.drop(columns=["timestamp"]).drop_duplicates("date")
    return df.set_index("date")


# ==============================
# Build dataframe for top 500 tokens
# ==============================
def build_top500_price_df(df_top500):
    """
    For each token in df_top500 fetch daily price data and merge into single DataFrame.
    Columns use CoinGecko IDs instead of symbols.
    """
    price_dfs = []
    for i, row in df_top500.iterrows():
        coin_id = row["id"]

        print(f"Fetching {coin_id} ...")
        df_price = get_historical_prices(coin_id, start_date="2015-01-01")
        if df_price is not None:
            df_price.rename(columns={"price": coin_id}, inplace=True)
            price_dfs.append(df_price)

        time.sleep(1)  # avoid hitting rate limits

    if not price_dfs:
        return None

    df_all = pd.concat(price_dfs, axis=1)
    return df_all

df_top500 = get_top_500_coins_table()
df_prices_top_500 = build_top500_price_df(df_top500)

from google.colab import drive
drive.mount('/content/drive')

# Save to your Google Drive
_export_dir = Path('/content/drive/MyDrive/Crypto 20 ETF/All_in_one/Tech/Tech_data')
df_prices_top_500.to_csv(_export_dir / "top500_prices_2015_to_today_27_10_2025.csv", index=True)

print("\n=== Combined Price Data (first 10 rows) ===")
print(df_prices_top_500.head(10))

Fetching bitcoin ...
Fetching ethereum ...
Fetching binancecoin ...
Fetching ripple ...
Fetching solana ...
Fetching tron ...
Fetching figure-heloc ...
Fetching dogecoin ...
Fetching whitebit ...
Fetching cardano ...
Fetching bitcoin-cash ...
Fetching hyperliquid ...
Fetching leo-token ...
Fetching monero ...
Fetching chainlink ...
Fetching canton-network ...
Fetching stellar ...
Fetching rain ...
Fetching litecoin ...
Fetching avalanche-2 ...
Fetching hedera-hashgraph ...
Fetching sui ...
Fetching zcash ...
Fetching shiba-inu ...
Fetching the-open-network ...
Fetching crypto-com-chain ...
Fetching world-liberty-financial ...
Fetching memecore ...
Fetching polkadot ...
Fetching uniswap ...
Fetching mantle ...
Fetching pi-network ...
Fetching hashnote-usyc ...
Fetching okb ...
Fetching bittensor ...
Fetching sky ...
Fetching aster-2 ...
Fetching near ...
Fetching aave ...
Fetching bitget-token ...
Fetching htx-dao ...
Fetching internet-computer ...
Fetching pepe ...
Fetching ethereum-cl

# **Get Volume Data**


In [ ]:
import requests
import pandas as pd
import datetime as dt
from tabulate import tabulate
import numpy as np
import time
import matplotlib.pyplot as plt
from pathlib import Path

API_KEY = "your-coingecko-pro-key-here"
headers = {
    "accept": "application/json",
    "x-cg-pro-api-key": API_KEY
}

In [ ]:
# ==============================
# STEP 1: Get Top 500 Tokens Today
# ==============================
def get_top_500_coins_table():
    url = "https://pro-api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": 250,
    }

    coins = []
    for page in range(1, 4):  # up to 750
        params["page"] = page
        response = requests.get(url, headers=headers, params=params)
        data = response.json()
        coins.extend(data)

    # Exclusion keywords
    exclude_keywords = [
        "usd", "usdt", "usdc", "busd", "dai", "tusd", "usdp", "gusd", "lusd", "fdusd",
        "usdd", "susd", "eusd", "wrapped", "wbtc", "weth", "renbtc", "staked", "stake"
    ]
    exclude_ids = [
        "bridged-wrapped-ether-starkgate","sbtc-2","wrapped-zenbtc","liquid-hype-yield",
        "compound-ether","binance-peg-sol","bitcoin-avalanche-bridged-btc-b","binance-peg-dogecoin",
        "tbtc","clbtc","tether-gold","rocket-pool-eth","solv-btc","pax-gold",
        "cgeth-hashkey","frax-ether","resolv-usr","jupiter-perpetual","gho", "stasis-eurs", "dola-usd", "blockchain-capital",
        "ousg", "mbg-by-multibank-group", "tradable-na-rent-financing-platform-sstn", "kinesis-gold", "kinesis-silver", "spiko-us-t-bills-money-market-fund",
        "onyc", "tradable-singapore-fintech-ssl-2", "vaneck-treasury-fund"
    ]

    def is_excluded(coin):
        name = coin["name"].lower()
        symbol = coin["symbol"].lower()
        cid = coin["id"].lower()
        return (any(kw in name or kw in symbol for kw in exclude_keywords) or cid in exclude_ids)

    included = [c for c in coins if not is_excluded(c)]

    df_included = pd.DataFrame(included)[["id", "symbol", "name", "current_price", "market_cap"]].head(500)
    df_included["market_cap_millions"] = (df_included["market_cap"] / 1_000_000).round(2)

    return df_included

In [ ]:
# ==============================
# Fetch historical daily trading volumes
# ==============================
def get_historical_volumes(coin_id, start_date="2015-01-01", end_date=None):
    """
    Fetch daily historical 24h trading volumes for a token from CoinGecko
    between start_date and end_date.
    Returns a DataFrame with [date, volume].
    """
    if end_date is None:
        end_date = dt.date.today().strftime("%Y-%m-%d")

    # Convert to UNIX timestamps
    start_ts = int(dt.datetime.strptime(start_date, "%Y-%m-%d").timestamp())
    end_ts = int(dt.datetime.strptime(end_date, "%Y-%m-%d").timestamp())

    url = f"https://pro-api.coingecko.com/api/v3/coins/{coin_id}/market_chart/range"
    params = {
        "vs_currency": "usd",
        "from": start_ts,
        "to": end_ts
    }

    r = requests.get(url, headers=headers, params=params)
    if r.status_code != 200:
        return None

    data = r.json()
    if "total_volumes" not in data:
        return None

    df = pd.DataFrame(data["total_volumes"], columns=["timestamp", "volume"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms").dt.date
    df = df.drop(columns=["timestamp"]).drop_duplicates("date")
    return df.set_index("date")


# ==============================
# Build dataframe for top 500 tokens (volumes)
# ==============================
def build_top500_volume_df(df_top500):
    """
    For each token in df_top500 fetch daily trading volume data
    and merge into a single DataFrame.
    Columns use CoinGecko IDs instead of symbols.
    """
    volume_dfs = []
    for i, row in df_top500.iterrows():
        coin_id = row["id"]

        print(f"Fetching volume for {coin_id} ...")
        df_volume = get_historical_volumes(coin_id, start_date="2015-01-01")
        if df_volume is not None:
            df_volume.rename(columns={"volume": coin_id}, inplace=True)
            volume_dfs.append(df_volume)

        time.sleep(1)  # avoid hitting rate limits

    if not volume_dfs:
        return None

    df_all = pd.concat(volume_dfs, axis=1)
    return df_all


# ==============================
# Run and save results
# ==============================
df_top500 = get_top_500_coins_table()
df_volumes_top_500 = build_top500_volume_df(df_top500)

from google.colab import drive
drive.mount('/content/drive')

# Save to your Google Drive
_export_dir = Path('/content/drive/MyDrive/Crypto 20 ETF/All_in_one/Tech/Tech_data')
df_volumes_top_500.to_csv(_export_dir / "top500_volumes_2015_to_today_27_10_2025.csv", index=True)

print("\n=== Combined Volume Data (first 10 rows) ===")
print(df_volumes_top_500.head(10))

Fetching volume for bitcoin ...
Fetching volume for ethereum ...
Fetching volume for binancecoin ...
Fetching volume for ripple ...
Fetching volume for solana ...
Fetching volume for tron ...
Fetching volume for figure-heloc ...
Fetching volume for dogecoin ...
Fetching volume for whitebit ...
Fetching volume for cardano ...
Fetching volume for bitcoin-cash ...
Fetching volume for leo-token ...
Fetching volume for hyperliquid ...
Fetching volume for monero ...
Fetching volume for chainlink ...
Fetching volume for canton-network ...
Fetching volume for stellar ...
Fetching volume for rain ...
Fetching volume for hedera-hashgraph ...
Fetching volume for litecoin ...
Fetching volume for avalanche-2 ...
Fetching volume for zcash ...
Fetching volume for sui ...
Fetching volume for shiba-inu ...
Fetching volume for crypto-com-chain ...
Fetching volume for the-open-network ...
Fetching volume for world-liberty-financial ...
Fetching volume for polkadot ...
Fetching volume for uniswap ...
Fetc

# **Get MCAP Data**

In [ ]:
# ==============================
# STEP 1: Get Top 500 Tokens Today
# ==============================
def get_top_500_coins_table():
    url = "https://pro-api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": 250,
    }

    coins = []
    for page in range(1, 4):  # up to 750
        params["page"] = page
        response = requests.get(url, headers=headers, params=params)
        data = response.json()
        coins.extend(data)

    # Exclusion keywords
    exclude_keywords = [
        "usd", "usdt", "usdc", "busd", "dai", "tusd", "usdp", "gusd", "lusd", "fdusd",
        "usdd", "susd", "eusd", "wrapped", "wbtc", "weth", "renbtc", "staked", "stake"
    ]
    exclude_ids = [
        "bridged-wrapped-ether-starkgate","sbtc-2","wrapped-zenbtc","liquid-hype-yield",
        "compound-ether","binance-peg-sol","bitcoin-avalanche-bridged-btc-b","binance-peg-dogecoin",
        "tbtc","clbtc","tether-gold","rocket-pool-eth","solv-btc","pax-gold",
        "cgeth-hashkey","frax-ether","resolv-usr","jupiter-perpetual","gho", "stasis-eurs", "dola-usd", "blockchain-capital",
        "ousg", "mbg-by-multibank-group", "tradable-na-rent-financing-platform-sstn", "kinesis-gold", "kinesis-silver", "spiko-us-t-bills-money-market-fund",
        "onyc", "tradable-singapore-fintech-ssl-2", "vaneck-treasury-fund"
    ]

    def is_excluded(coin):
        name = coin["name"].lower()
        symbol = coin["symbol"].lower()
        cid = coin["id"].lower()
        return (any(kw in name or kw in symbol for kw in exclude_keywords) or cid in exclude_ids)

    included = [c for c in coins if not is_excluded(c)]

    df_included = pd.DataFrame(included)[["id", "symbol", "name", "current_price", "market_cap"]].head(500)
    df_included["market_cap_millions"] = (df_included["market_cap"] / 1_000_000).round(2)

    return df_included

In [ ]:
# ==============================
# Fetch historical daily market caps
# ==============================
def get_historical_mcap(coin_id, start_date="2015-01-01", end_date=None):
    """
    Fetch daily historical market cap for a token from CoinGecko between start_date and end_date.
    Returns a DataFrame with [date, market_cap].
    """
    if end_date is None:
        end_date = dt.date.today().strftime("%Y-%m-%d")

    # Convert to UNIX timestamps
    start_ts = int(dt.datetime.strptime(start_date, "%Y-%m-%d").timestamp())
    end_ts = int(dt.datetime.strptime(end_date, "%Y-%m-%d").timestamp())

    url = f"https://pro-api.coingecko.com/api/v3/coins/{coin_id}/market_chart/range"
    params = {
        "vs_currency": "usd",
        "from": start_ts,
        "to": end_ts
    }

    r = requests.get(url, headers=headers, params=params)
    if r.status_code != 200:
        print(f"Error fetching {coin_id}: {r.status_code}")
        return None

    data = r.json()
    if "market_caps" not in data:
        print(f"No market cap data for {coin_id}")
        return None

    df = pd.DataFrame(data["market_caps"], columns=["timestamp", "market_cap"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms").dt.date
    df = df.drop(columns=["timestamp"]).drop_duplicates("date")
    return df.set_index("date")


# ==============================
# Build dataframe for top 500 tokens
# ==============================
def build_top500_mcap_df(df_top500):
    """
    For each token in df_top100 fetch daily market cap data and merge into single DataFrame.
    Columns use CoinGecko IDs instead of symbols.
    """
    mcap_dfs = []
    for i, row in df_top500.iterrows():
        coin_id = row["id"]

        print(f"Fetching market cap for {coin_id} ...")
        df_mcap = get_historical_mcap(coin_id, start_date="2015-01-01")
        if df_mcap is not None:
            df_mcap.rename(columns={"market_cap": coin_id}, inplace=True)
            mcap_dfs.append(df_mcap)

        time.sleep(1)  # avoid hitting rate limits

    if not mcap_dfs:
        return None

    df_all = pd.concat(mcap_dfs, axis=1)
    return df_all


df_top500 = get_top_500_coins_table()
df_mcaps = build_top500_mcap_df(df_top500)

from google.colab import drive
drive.mount('/content/drive')

_export_dir = Path('/content/drive/MyDrive/Crypto 20 ETF/All_in_one/Tech/Tech_data')
df_mcaps.to_csv(_export_dir / "top500_mcap_2015_to_today_27_10_2025.csv", index=True)


print("\n=== Combined Market Cap Data (first 10 rows) ===")
print(df_mcaps.head(10))

Fetching market cap for bitcoin ...
Fetching market cap for ethereum ...
Fetching market cap for binancecoin ...
Fetching market cap for ripple ...
Fetching market cap for solana ...
Fetching market cap for tron ...
Fetching market cap for figure-heloc ...
Fetching market cap for dogecoin ...
Fetching market cap for whitebit ...
Fetching market cap for cardano ...
Fetching market cap for bitcoin-cash ...
Fetching market cap for leo-token ...
Fetching market cap for hyperliquid ...
Fetching market cap for monero ...
Fetching market cap for chainlink ...
Fetching market cap for canton-network ...
Fetching market cap for stellar ...
Fetching market cap for rain ...
Fetching market cap for litecoin ...
Fetching market cap for hedera-hashgraph ...
Fetching market cap for avalanche-2 ...
Fetching market cap for zcash ...
Fetching market cap for sui ...
Fetching market cap for shiba-inu ...
Fetching market cap for crypto-com-chain ...
Fetching market cap for the-open-network ...
Fetching mark

In [ ]:
# ============================================================================
# Rolling Training Framework for Crypto
# ============================================================================
# 
# Rolling training framework based on existing BacktestEngine
# 
# Architecture:
# ┌──────────────────────────────────────────────────────────────┐
# │                   Rolling Train-Test Split                    │
# │  ┌─────────┐  ┌─────────┐  ┌─────────┐  ┌─────────┐         │
# │  │ Train 1 │->│ Test 1  │->│ Train 2 │->│ Test 2  │ ...     │
# │  └─────────┘  └─────────┘  └─────────┘  └─────────┘         │
# └──────────────────────────────────────────────────────────────┘
#
# ============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta            
from typing import Dict, List, Callable, Optional, Tuple, Any
import warnings
import copy
warnings.filterwarnings('ignore')

# ============================================================================
# PART 1: Data Adapter
# ============================================================================


class CryptoDataAdapter:
    """
    Convert existing data format to rolling training format
    
    Input: df_prices, df_mcap, df_fees, df_dau (wide format)
    Output: Feature matrix + label vector
    """
    
    def __init__(self, df_prices, df_mcap, df_fees=None, df_dau=None, df_volume=None,
                 base_universe: list = None,
                 mcap_topk_filter: int = None):
        self.df_prices = df_prices.copy()
        self.df_mcap = df_mcap.copy()
        self.df_fees = df_fees.copy() if df_fees is not None else None
        self.df_dau = df_dau.copy() if df_dau is not None else None
        self.df_volume = df_volume.copy() if df_volume is not None else None

        # 动态候选池（先在Excel池子里，再按每期 mcap TopK 过滤）
        # NOTE: base_universe 一般就是 apply_target_universe() 的返回（Excel顺序，id已lower）
        try:
            self.base_universe = [str(x).strip().lower() for x in (base_universe or []) if str(x).strip()]
        except Exception:
            self.base_universe = []

        try:
            self.mcap_topk_filter = None if mcap_topk_filter is None else int(mcap_topk_filter)
            if self.mcap_topk_filter is not None and self.mcap_topk_filter <= 0:
                self.mcap_topk_filter = None
        except Exception:
            self.mcap_topk_filter = None

        # Fill-gap counters (for single-factor / feature audit)
        self._fillgap_stats = {'price': 0, 'volume': 0}

        # Standardize index format
        for df in [self.df_prices, self.df_mcap]:
            if not isinstance(df.index, pd.DatetimeIndex):
                df.index = pd.to_datetime(df.index)

        if self.df_volume is not None and (not isinstance(self.df_volume.index, pd.DatetimeIndex)):
            self.df_volume.index = pd.to_datetime(self.df_volume.index)

        # Cache month-end trading days for monthly label/splitting alignment
        try:
            idx = self.df_prices.index
            month_end = idx.to_series().groupby(idx.to_period('M')).max()
            self._month_end_dates = pd.DatetimeIndex(month_end.values).sort_values()
        except Exception:
            self._month_end_dates = pd.DatetimeIndex([])
        
        # ============================================================
        # Cache decad-end trading days for decad (旬) label/splitting alignment
        # 旬末日期：每月10日、20日、月末对应的最近交易日
        # ============================================================
        try:
            self._decad_end_dates = self._compute_decad_end_dates(self.df_prices.index)
        except Exception:
            self._decad_end_dates = pd.DatetimeIndex([])
    
    def _compute_decad_end_dates(self, calendar: pd.DatetimeIndex) -> pd.DatetimeIndex:
        """
        计算所有旬末交易日 / Compute all decad-end trading days
        
        旬的定义:
        - 上旬 (D1): 1日-10日, 旬末 = 10日或之前最近的交易日
        - 中旬 (D2): 11日-20日, 旬末 = 20日或之前最近的交易日
        - 下旬 (D3): 21日-月末, 旬末 = 月末最后一个交易日
        """
        if calendar is None or len(calendar) == 0:
            return pd.DatetimeIndex([])
        
        calendar = calendar.sort_values()
        decad_ends = []
        
        # 按年月分组处理
        df_cal = pd.DataFrame({'date': calendar})
        df_cal['year'] = df_cal['date'].dt.year
        df_cal['month'] = df_cal['date'].dt.month
        df_cal['day'] = df_cal['date'].dt.day
        
        for (year, month), group in df_cal.groupby(['year', 'month']):
            days = group['date'].sort_values()
            
            # 上旬末 (<=10日的最大日期)
            d1_dates = days[days.dt.day <= 10]
            if len(d1_dates) > 0:
                decad_ends.append(d1_dates.iloc[-1])
            
            # 中旬末 (11-20日的最大日期)
            d2_dates = days[(days.dt.day >= 11) & (days.dt.day <= 20)]
            if len(d2_dates) > 0:
                decad_ends.append(d2_dates.iloc[-1])
            
            # 下旬末 (21日及之后的最大日期 = 月末)
            d3_dates = days[days.dt.day >= 21]
            if len(d3_dates) > 0:
                decad_ends.append(d3_dates.iloc[-1])
        
        return pd.DatetimeIndex(decad_ends).sort_values()
    
    def _next_decad_end(self, date: pd.Timestamp, forward_decads: int = 1) -> Optional[pd.Timestamp]:
        """返回 date 之后第 forward_decads 个"旬末交易日" / Next decad-end trading day"""
        if self._decad_end_dates is None or len(self._decad_end_dates) == 0:
            return None

        d = pd.to_datetime(date)
        future = self._decad_end_dates[self._decad_end_dates > d]
        if len(future) < forward_decads:
            return None
        return future[forward_decads - 1]
    

    #调试1    
    def compute_features(self, date, lookback_days=120, universe_size=100):
        """
        计算指定日期的特征矩阵 - 增强版 (含技术指标)
        
        Features:
        基础特征（去掉纯动量/短期涨幅类）：
        - vol_20d, vol_60d: 波动率
        - log_mcap: 对数市值

        技术指标特征:
        - sma_10, sma_20, sma_50: 简单移动平均（价/SMA - 1）
        - sma_ratio_10_50: SMA比率 (短期/长期)
        - ema_10, ema_20: EMA偏离（价/EMA - 1）
        - rma_10, rma_20: RMA(Wilder)偏离（价/RMA - 1）
        - rsi_14: RSI相对强弱指数（Wilder RMA平滑）
        - macd, macd_signal, macd_hist: MACD指标（归一化到价格）
        - bb_pctb, bb_width: 布林带 %B 与 带宽
        - mean_reversion_20, mean_reversion_60: 均值恢复信号(-Z)
        """
        available_dates = self.df_prices.index[self.df_prices.index <= date]
        if len(available_dates) < lookback_days + 1:
            return None
        
        end_date = available_dates[-1]
        
        # 选取universe
        mcaps = self.df_mcap.loc[end_date].dropna()
        if mcaps.empty:
            return None

        # 先在 Excel 池子里取（如果你前面 apply_target_universe 已经裁剪过，这里基本等价于全量）
        try:
            if isinstance(getattr(self, 'base_universe', None), list) and len(self.base_universe) > 0:
                _bu = [x for x in self.base_universe if x in mcaps.index]
                if len(_bu) > 0:
                    mcaps = mcaps.reindex(_bu).dropna()
        except Exception:
            pass

        # 再做动态 mcap TopK 过滤（每个锚点日单独计算）
        try:
            _topk = getattr(self, 'mcap_topk_filter', None)
            if _topk is not None:
                _topk = int(_topk)
                if _topk > 0 and _topk < len(mcaps):
                    mcaps = mcaps.nlargest(_topk)
        except Exception:
            pass

        # 最后再按 universe_size 截断（仍然是按当日 mcap 排序）
        try:
            _u = None if universe_size is None else int(universe_size)
        except Exception:
            _u = None

        if mcaps.empty:
            return None

        if _u is not None and _u > 0 and _u < len(mcaps):
            mcaps = mcaps.nlargest(_u)

        assets = mcaps.index

        # ============================================================
        # 数据口径硬门槛：最小历史长度（防止新币上线噪声/缺失被ffill“伪造”为稳定价格）
        #
        # 科学原则：资产在锚点日 end_date 必须满足：
        #   1) >= 30 个有效交易日“真实价格点”（原始 df_prices 非NaN，不依赖 ffill）
        #   2) >= 30 日历天上市期（first_valid_price -> end_date）
        # 不满足则该币在该锚点日不参与：训练/预测/交易。
        # ============================================================
        # 口径：新币准入门槛（不要因为中间缺 1 天就整币踢掉）
        # 中文: 只处理“上市太晚(first_valid太靠后)”这种；中间缺口后面会用 ffill_between_valid 修复。
        MIN_LISTING_CALENDAR_DAYS = int(globals().get('MIN_LISTING_CALENDAR_DAYS', 30))
        MIN_REAL_PRICE_POINTS = int(globals().get('MIN_REAL_PRICE_POINTS', 10))

        raw_prices = self.df_prices.loc[:end_date, assets]  # 注意：这里不做ffill，用于统计“真实价格点”
        eligible_assets = []
        for a in list(assets):
            s = raw_prices[a]
            first_valid = s.first_valid_index()
            if first_valid is None:
                continue

            # 上市期（日历天）
            listing_days = (pd.to_datetime(end_date) - pd.to_datetime(first_valid)).days
            if listing_days < MIN_LISTING_CALENDAR_DAYS:
                continue

            # 真实价格点：不要求“每天都有”，只要不是稀疏到完全算不出东西
            real_points = int(s.loc[first_valid:end_date].notna().sum())
            if real_points < MIN_REAL_PRICE_POINTS:
                continue

            eligible_assets.append(a)

        assets = pd.Index(eligible_assets)
        if len(assets) == 0:
            return None
        
                # 获取价格序列（仅对通过准入过滤的资产）
        # - 缺价格：视作“价格不变”（用历史信息前向填充），这样当日变动自然为0
        # - 只填内部缺口：要求前后都有数据（避免把尾部断档也硬填）
        # - 不做 bfill：避免引入未来信息
        prices_raw = self.df_prices.loc[:end_date, assets].dropna(how='all')

        # NOTE: 人类习惯的小工具，避免上市后“中间缺口”把技术指标搞歪
        # - 返回 (filled_df, n_filled_points)
        def _ffill_between_valid_df(df: pd.DataFrame):
            if df is None or len(df) == 0:
                return df, 0

            out = df.copy()
            n_rep = 0
            for c in list(out.columns):
                s = out[c]
                fv = s.first_valid_index()
                lv = s.last_valid_index()
                if fv is None or lv is None:
                    continue

                seg = s.loc[fv:lv]
                n_rep += int(seg.isna().sum())
                out.loc[fv:lv, c] = seg.ffill()

            return out, int(n_rep)

        prices, _n_fill_prices = _ffill_between_valid_df(prices_raw)

        # 累计统计：给单因子/研究面板打印
        if hasattr(self, '_fillgap_stats') and isinstance(self._fillgap_stats, dict):
            self._fillgap_stats['price'] = int(self._fillgap_stats.get('price', 0)) + int(_n_fill_prices)

                # 安全边界：锚点日当天仍然缺价的资产不参与（不能用未来数据去补当天）
        prices = prices.dropna(axis=1, how='all')
        if prices is None or len(prices.columns) == 0:
            return None

        # 如果某资产在 end_date 当天是 NaN：直接剔除
        end_px = prices.iloc[-1]
        bad_cols = list(end_px[end_px.isna()].index)
        if len(bad_cols) > 0:
            prices = prices.drop(columns=bad_cols)

        assets = list(prices.columns)
        if len(assets) == 0:
            return None

        # 成交量也做同样处理（如果有），并把“修复后的成交量窗口”传给后续特征计算
        vol_df_local = None
        _n_fill_volume = 0
        if self.df_volume is not None:
            vol_raw = self.df_volume.loc[prices.index, assets].dropna(how='all')
            vol_df_local, _n_fill_volume = _ffill_between_valid_df(vol_raw)
            if hasattr(self, '_fillgap_stats') and isinstance(self._fillgap_stats, dict):
                self._fillgap_stats['volume'] = int(self._fillgap_stats.get('volume', 0)) + int(_n_fill_volume)

        # 可选调试：避免刷屏
        if bool(globals().get('FILL_GAP_VERBOSE', False)):
            print(f"[FillGap@features] {pd.to_datetime(end_date).date()} price_fill={_n_fill_prices} volume_fill={_n_fill_volume} dropped_end_na={len(bad_cols)}")

        if len(prices) < lookback_days:
            return None

        features = {}

        def _clean(x):
            """统一把 NaN/inf 清理为 0（便于统计/画图；训练阶段也更稳定）"""
            if isinstance(x, (pd.Series, pd.DataFrame)):
                return x.replace([np.inf, -np.inf], 0).fillna(0)
            try:
                # numpy scalar / python float
                if x is None:
                    return 0.0
                if isinstance(x, (float, int)) and (not np.isfinite(x)):
                    return 0.0
            except Exception:
                pass
            return x
        
        # ============================================================
        # Part 1: 基础特征（删除“短期涨幅/动量类”列）
        # ============================================================
        # 波动率
        # 缺价格已做 ffill，因此 pct_change 在缺失段为0；首行 NaN 也按0处理
        rets_daily = prices.pct_change().fillna(0)
        for days in [20, 60]:
            if len(rets_daily) >= days:
                vol = rets_daily.iloc[-days:].std() * np.sqrt(365)
                features[f'vol_{days}d'] = vol

        # 对数市值
        features['log_mcap'] = np.log(mcaps[assets] + 1)


        # ============================================================
        # Part 2: SMA/EMA/RMA - 多参数移动平均 + 交叉接近度
        # ============================================================
        eps = 1e-8

        # 多参数窗口（回到更“像交易员”的尺度：短/中/长都覆盖，但不铺太满）
        # NOTE: 10/50 必须保留（兼容 sma_ratio_10_50 等旧字段）
        SMA_WINDOWS = [5, 7, 9, 10, 11, 13, 20, 30, 40, 50, 60]
        EMA_WINDOWS = [5, 7, 9, 10, 11, 13, 20, 30, 40, 50, 60]
        RMA_WINDOWS = [10, 20]

        # ---- SMA 偏离（P/SMA - 1）----
        sma_val = {}
        for w in SMA_WINDOWS:
            if len(prices) >= w:
                sma_w = prices.iloc[-w:].mean()
                sma_val[w] = sma_w
                features[f'sma_{w}'] = (prices.iloc[-1] / (sma_w + eps) - 1)

        # 保留原有字段（兼容旧代码）
        # 语义修正：sma_ratio_* 应该表示“短期/长期”的相对比率
        if 10 in sma_val and 50 in sma_val:
            features['sma_ratio_10_50'] = (sma_val[10] / (sma_val[50] + eps) - 1).replace([np.inf, -np.inf], 0).fillna(0)
            # 旧实现是两个“偏离(P/SMA-1)”之差： (P/SMA10-1) - (P/SMA50-1)
            # 这里必须保持为按资产的向量（pd.Series），不能转 float
            dev10 = features.get('sma_10')
            dev50 = features.get('sma_50')
            if not isinstance(dev10, pd.Series):
                dev10 = pd.Series(0.0, index=assets)
            if not isinstance(dev50, pd.Series):
                dev50 = pd.Series(0.0, index=assets)
            features['sma_devdiff_10_50'] = (dev10 - dev50).replace([np.inf, -np.inf], 0).fillna(0)

        # ---- EMA 偏离（P/EMA - 1）----（向量化，避免逐币循环）
        ema_val = {}
        for w in EMA_WINDOWS:
            if len(prices) >= max(w, 20):
                ema_w = prices.ewm(span=w, adjust=False).mean().iloc[-1]
                ema_val[w] = ema_w
                features[f'ema_{w}'] = (prices.iloc[-1] / (ema_w + eps) - 1)

        # ---- RMA 偏离（P/RMA - 1，Wilder smoothing）----
        for w in RMA_WINDOWS:
            if len(prices) >= max(w, 20):
                rma_w = prices.ewm(alpha=1.0 / w, adjust=False).mean().iloc[-1]
                features[f'rma_{w}'] = (prices.iloc[-1] / (rma_w + eps) - 1)

        # ============================================================
        # Part 2.5: 价格斜率（正斜率门控，用于交叉信号强度）
        # ============================================================
        SLOPE_WINDOWS = [5, 20]
        for k in SLOPE_WINDOWS:
            if len(prices) >= k:
                tail = prices.tail(k)
                y = np.log(tail + eps).values  # (k, n_assets)
                x = np.arange(k, dtype=float)
                x = x - x.mean()
                denom = (x ** 2).sum() + eps
                slope = (y * x.reshape(-1, 1)).sum(axis=0) / denom
                slope_s = pd.Series(slope, index=tail.columns)
                features[f'slope_logp_{k}'] = slope_s
                features[f'gate_slope_pos_{k}'] = (slope_s > 0).astype(float)

        # ============================================================
        # Part 2.6: SMA/EMA 交叉特征（接近度 + 正斜率门控）
        # ============================================================
        def _prox_from_diff(diff: pd.Series, scale: float = 0.01) -> pd.Series:
            """Cross proximity mapping.

            diff is normalized by price (roughly a % spread). This returns a smooth
            proximity score in (0, 1], where 1 means very close to crossover (diff≈0).

            Note: using a rational form avoids the hard saturation of tanh(1/|diff|).
            """
            x = diff.abs().replace([np.inf, -np.inf], np.nan)
            prox = 1.0 / (1.0 + x / (scale + 1e-12))
            return prox.fillna(0).clip(0, 1)

        gate10 = features.get('gate_slope_pos_10')
        if gate10 is None:
            gate10 = pd.Series(0.0, index=assets)

        # 交叉对（按你说的规律：fast 每次 +2；slow 每次 +10）
        CROSS_PAIRS = [(5, 20), (7, 30), (9, 40), (11, 50), (13, 60)]

        for fast, slow in CROSS_PAIRS:
            # --- SMA cross ---
            if fast in sma_val and slow in sma_val:
                diff = (sma_val[fast] - sma_val[slow]) / (prices.iloc[-1] + eps)
                prox = _prox_from_diff(diff)
                features[f'sma_diff_{fast}_{slow}'] = diff
                features[f'sma_prox_{fast}_{slow}'] = prox

                # 无符号强度：仅表示“接近交叉 + 正斜率门控”
                features[f'sma_cross_strength_{fast}_{slow}'] = prox * gate10
                # 有符号强度：保留金叉/死叉方向（更适合模型学习方向性）
                features[f'sma_cross_strength_signed_{fast}_{slow}'] = (prox * np.sign(diff) * gate10).replace([np.inf, -np.inf], 0).fillna(0)

                # 事件型特征：是否“刚发生交叉”（参考 kdj_T_1.py 的 crossings 思路，但做成 ML 特征）
                # 用 rolling SMA 取上一天的diff符号，避免只看静态状态
                if len(prices) >= slow + 2:
                    sma_fast_series = prices.rolling(window=fast, min_periods=fast).mean()
                    sma_slow_series = prices.rolling(window=slow, min_periods=slow).mean()
                    diff_series = (sma_fast_series - sma_slow_series) / (prices + eps)
                    diff_prev = diff_series.iloc[-2].replace([np.inf, -np.inf], 0).fillna(0)
                    diff_last = diff_series.iloc[-1].replace([np.inf, -np.inf], 0).fillna(0)
                    cross_up = ((diff_prev <= 0) & (diff_last > 0)).astype(float)
                    cross_down = ((diff_prev >= 0) & (diff_last < 0)).astype(float)
                    # features[f'sma_cross_up_{fast}_{slow}'] = cross_up
                    features[f'sma_cross_down_{fast}_{slow}'] = cross_down
                    # signed event: +1 上穿(金叉), -1 下穿(死叉)
                    # features[f'sma_cross_event_{fast}_{slow}'] = (cross_up - cross_down)

            # --- EMA cross ---
            if fast in ema_val and slow in ema_val:
                diff = (ema_val[fast] - ema_val[slow]) / (prices.iloc[-1] + eps)
                prox = _prox_from_diff(diff)
                features[f'ema_diff_{fast}_{slow}'] = diff
                features[f'ema_prox_{fast}_{slow}'] = prox

                features[f'ema_cross_strength_{fast}_{slow}'] = prox * gate10
                features[f'ema_cross_strength_signed_{fast}_{slow}'] = (prox * np.sign(diff) * gate10).replace([np.inf, -np.inf], 0).fillna(0)

                if len(prices) >= slow + 2:
                    ema_fast_series = prices.ewm(span=fast, adjust=False).mean()
                    ema_slow_series = prices.ewm(span=slow, adjust=False).mean()
                    diff_series = (ema_fast_series - ema_slow_series) / (prices + eps)
                    diff_prev = diff_series.iloc[-2].replace([np.inf, -np.inf], 0).fillna(0)
                    diff_last = diff_series.iloc[-1].replace([np.inf, -np.inf], 0).fillna(0)
                    cross_up = ((diff_prev <= 0) & (diff_last > 0)).astype(float)
                    cross_down = ((diff_prev >= 0) & (diff_last < 0)).astype(float)
                    # features[f'ema_cross_up_{fast}_{slow}'] = cross_up
                    features[f'ema_cross_down_{fast}_{slow}'] = cross_down
                    # features[f'ema_cross_event_{fast}_{slow}'] = (cross_up - cross_down)

        # ============================================================
        # Part 3: RSI - 多参数（Wilder RMA平滑）
        # ============================================================
        # 用日收益的变化来计算 RSI（close-only）
        delta = prices.diff()
        gains = delta.clip(lower=0)
        losses = (-delta).clip(lower=0)

        RSI_PERIODS = [7, 14, 21, 28]
        RSI_limit = [30, 70]
        rsi_low, rsi_high = RSI_limit
        for p in RSI_PERIODS:
            if len(prices) >= p + 5:
                # 全序列 RSI（Wilder/RMA），便于提取 turning/crossing 事件
                avg_gain_series = gains.ewm(alpha=1.0 / p, adjust=False).mean()
                avg_loss_series = losses.ewm(alpha=1.0 / p, adjust=False).mean()
                rs_series = avg_gain_series / (avg_loss_series + eps)
                rsi_series = 100 - (100 / (1 + rs_series))

                rsi = rsi_series.iloc[-1]

                # RSI level：标准化到 [-1, 1]
                rsi_scaled = (rsi - 50) / 50
                rsi_scaled = rsi_scaled.replace([np.inf, -np.inf], 0).fillna(0)
                features[f'rsi_{p}'] = rsi_scaled

                # RSI family（更像“抄底回归”但保留动量表达）
                # 关键改动：不要用 clip(lower=0) 把大多数币压成 0（横截面同分 -> TopN同权）
                # 思路：
                # - 连续距离：离 30/70 有多远（可正可负）
                # - 软阈值：用 sigmoid 平滑激活“超卖/超买状态”，避免离散0/1
                # - 回升强度：从近期RSI低点回升多少（bottoming），同时用 RSI 自身动量过滤“还在下跌的超卖”

                # 1) 连续距离（可排序；RankIC 正/负会自动决定更像动量还是反转）
                rsi_dist_os = (rsi_low - rsi) / rsi_low          # >0: oversold, <0: not-oversold
                rsi_dist_ob = (rsi - rsi_high) / rsi_low          # >0: overbought, <0: not-overbought
                features[f'rsi_dist_os_{p}'] = rsi_dist_os.replace([np.inf, -np.inf], 0).fillna(0)
                features[f'rsi_dist_ob_{p}'] = rsi_dist_ob.replace([np.inf, -np.inf], 0).fillna(0)

                # v6.1 legacy 口径（恢复 _clip，用于你原来跑出来更好的那类结果）
                rsi_dist_os_clip = ((rsi_low - rsi) / rsi_low).clip(lower=0)
                rsi_dist_ob_clip = ((rsi - rsi_high) / rsi_low).clip(lower=0)
                features[f'rsi_dist_os_{p}_clip'] = rsi_dist_os_clip.replace([np.inf, -np.inf], 0).fillna(0)
                features[f'rsi_dist_ob_{p}_clip'] = rsi_dist_ob_clip.replace([np.inf, -np.inf], 0).fillna(0)

                # 2) 软阈值（状态强度，不是0/1）
                #    rsi_os_soft 接近 1 表示更“超卖”，接近 0 表示远离超卖
                _soft_s = 5.0
                rsi_os_soft = 1.0 / (1.0 + np.exp((rsi - rsi_low) / _soft_s))
                rsi_ob_soft = 1.0 / (1.0 + np.exp((rsi_high - rsi) / _soft_s))
                # features[f'rsi_os_soft_{p}'] = pd.Series(rsi_os_soft, index=rsi.index).replace([np.inf, -np.inf], 0).fillna(0)
                # features[f'rsi_ob_soft_{p}'] = pd.Series(rsi_ob_soft, index=rsi.index).replace([np.inf, -np.inf], 0).fillna(0)

                # 3) 回升强度 / 底部确认（C1温和删减：这两类在训练窗更容易成为噪声冠军，先删除以降低过拟合）

                # turning 事件：RSI vs RSI_3day_MA 的上穿/下穿（保留离散事件，同时补一个连续gap）
                rsi_ma3 = rsi_series.rolling(window=3, min_periods=3).mean()
                rsi_prev = rsi_series.iloc[-2]
                ma_prev = rsi_ma3.iloc[-2]
                rsi_last = rsi_series.iloc[-1]
                ma_last = rsi_ma3.iloc[-1]

                # C1温和删减：删除 RSI-MA3 gap 这类新增连续gap列（训练窗容易成为噪声冠军）

                turn_up = ((rsi_prev < ma_prev) & (rsi_last > ma_last)).astype(float)
                turn_down = ((rsi_prev > ma_prev) & (rsi_last < ma_last)).astype(float)
                # features[f'rsi_turn_up_{p}'] = turn_up.replace([np.inf, -np.inf], 0).fillna(0)
                # features[f'rsi_turn_down_{p}'] = turn_down.replace([np.inf, -np.inf], 0).fillna(0)
                # features[f'rsi_turn_event_{p}'] = (turn_up - turn_down).replace([np.inf, -np.inf], 0).fillna(0)

                # --- RSI_MR extra (continuous + turning confirm + state gate) ---
                try:
                    _os_soft = features.get(f'rsi_os_soft_{p}')
                    if not isinstance(_os_soft, pd.Series):
                        _os_soft = pd.Series(rsi_os_soft, index=rsi.index)
                except Exception:
                    _os_soft = pd.Series(0.0, index=rsi.index)

                # 1) 连续超卖强度（所有币都有值，减少 ties）
                # features[f'rsi_mr_soft_{p}'] = _os_soft.replace([np.inf, -np.inf], 0).fillna(0)

                # 2) 拐头确认：RSI 高于其 MA3 的“gap”（只取正半轴，表示由弱转强）
                turn_gap = (rsi_last - ma_last) / 50.0
                turn_gap = pd.Series(turn_gap, index=rsi.index).replace([np.inf, -np.inf], 0).fillna(0)
                # features[f'rsi_mr_softturn_{p}'] = (_os_soft * turn_gap.clip(lower=0)).replace([np.inf, -np.inf], 0).fillna(0)

                # 3) 状态过滤：叠加价格短期正斜率(5d)门控（更像“跌完开始抬头”的反转）
                gate5 = features.get('gate_slope_pos_5')
                if gate5 is None:
                    gate5 = pd.Series(0.0, index=rsi.index)
                if not isinstance(gate5, pd.Series):
                    gate5 = pd.Series(gate5, index=rsi.index)
                # features[f'rsi_mr_softturn_gate5_{p}'] = (_os_soft * turn_gap.clip(lower=0) * gate5).replace([np.inf, -np.inf], 0).fillna(0)

    

        # ============================================================
        # Part 4: MACD - 多参数族 + RMA平滑（来自你 kdj_T_1.py 思路）
        # ============================================================
        # MACD 参数组：按你指定的“更标准”的4组（配合4个口径=16列）
        MACD_PARAMS = [
            (5, 10, 4),
            (10, 20, 8),
            (12, 26, 9),
            (20, 40, 16),
        ]
        MACD_RMA_SMOOTH = [3]

        price_now = prices.iloc[-1]

        for fast, slow, sig in MACD_PARAMS:
            if len(prices) < max(slow + sig, 35):
                continue

            ema_fast = prices.ewm(span=fast, adjust=False).mean()
            ema_slow = prices.ewm(span=slow, adjust=False).mean()
            macd_line = ema_fast - ema_slow
            signal_line = macd_line.ewm(span=sig, adjust=False).mean()
            hist = macd_line - signal_line

            line_last = (macd_line.iloc[-1] / (price_now + eps)).replace([np.inf, -np.inf], 0).fillna(0)
            sig_last = (signal_line.iloc[-1] / (price_now + eps)).replace([np.inf, -np.inf], 0).fillna(0)
            hist_last = (hist.iloc[-1] / (price_now + eps)).replace([np.inf, -np.inf], 0).fillna(0)

            key = f'{fast}_{slow}_{sig}'
            features[f'macd_line_{key}'] = line_last
            features[f'macd_signal_{key}'] = sig_last
            features[f'macd_hist_{key}'] = hist_last

            # 接近零轴（强度）
            # features[f'macd_prox0_{key}'] = _prox_from_diff(line_last, scale=0.01)

            # 事件型特征：MACD 金叉/死叉（等价于 hist 穿越 0）
            # 这里用“归一化后的 hist”避免不同币价格尺度影响
            hist_norm = (hist / (prices + eps)).replace([np.inf, -np.inf], 0).fillna(0)
            if len(hist_norm) >= 2:
                hist_prev = hist_norm.iloc[-2]
                hist_now = hist_norm.iloc[-1]
                macd_cross_up = ((hist_prev <= 0) & (hist_now > 0)).astype(float)
                macd_cross_down = ((hist_prev >= 0) & (hist_now < 0)).astype(float)
                features[f'macd_cross_up_{key}'] = macd_cross_up
                features[f'macd_cross_down_{key}'] = macd_cross_down
                features[f'macd_cross_event_{key}'] = (macd_cross_up - macd_cross_down)

            # hist 的 RMA 平滑（去噪）
            for r in MACD_RMA_SMOOTH:
                hist_rma_last = (hist.ewm(alpha=1.0 / r, adjust=False).mean().iloc[-1] / (price_now + eps))
                hist_rma_last = hist_rma_last.replace([np.inf, -np.inf], 0).fillna(0)
                # features[f'macd_hist_rma{r}_{key}'] = hist_rma_last

            # hist 变化斜率（动能变化强度，close-only）
            if len(hist) >= 6:
                hist_norm = (hist / (prices + eps)).iloc[-6:]
                hist_slope5 = (hist_norm.iloc[-1] - hist_norm.iloc[0]) / 5
                # features[f'macd_hist_slope5_{key}'] = hist_slope5.replace([np.inf, -np.inf], 0).fillna(0)

        # 兼容旧字段：保留 (12,26,9) 的 macd/macd_signal/macd_hist
        if len(prices) >= 35:
            ema12 = prices.ewm(span=12, adjust=False).mean()
            ema26 = prices.ewm(span=26, adjust=False).mean()
            macd_line = ema12 - ema26
            signal_line = macd_line.ewm(span=9, adjust=False).mean()
            hist = macd_line - signal_line
            # features['macd'] = (macd_line.iloc[-1] / (price_now + eps)).replace([np.inf, -np.inf], 0).fillna(0)
            # features['macd_signal'] = (signal_line.iloc[-1] / (price_now + eps)).replace([np.inf, -np.inf], 0).fillna(0)
            # features['macd_hist'] = (hist.iloc[-1] / (price_now + eps)).replace([np.inf, -np.inf], 0).fillna(0)

        # ============================================================
        # Part 5: Bollinger Bands - 周期不要砍太狠（保持 ~20列）
        # ============================================================
        BB_PERIODS = [5, 10, 20, 40, 80]
        for p in BB_PERIODS:
            if len(prices) >= p:
                mid = prices.iloc[-p:].mean()
                std = prices.iloc[-p:].std()
                upper = mid + 2 * std
                lower = mid - 2 * std

                # %B（标准化到[-0.5,0.5]）
                pctb = (price_now - lower) / ((upper - lower) + eps) - 0.5
                pctb = pctb.replace([np.inf, -np.inf], 0).fillna(0)

                # 带宽
                width = ((upper - lower) / (mid + eps)).replace([np.inf, -np.inf], 0).fillna(0)

                # 中轨回归强度（Z）
                bb_z = ((price_now - mid) / (std + eps)).replace([np.inf, -np.inf], 0).fillna(0)

                # features[f'bb_pctb_{p}'] = pctb
                features[f'bb_width_{p}'] = width
                features[f'bb_z_{p}'] = bb_z

                # squeeze：带宽越小越“挤压”，用横截面标准化后的负值表达
                width_cs_z = ((width - width.mean()) / (width.std() + eps)).replace([np.inf, -np.inf], 0).fillna(0)
                # features[f'bb_squeeze_{p}'] = (-width_cs_z).replace([np.inf, -np.inf], 0).fillna(0)

        # 兼容旧字段：bb_pctb/bb_width 默认为 period=20
        if 'bb_pctb_20' in features:
            # features['bb_pctb'] = features['bb_pctb_20']
            pass
        if 'bb_width_20' in features:
            features['bb_width'] = features['bb_width_20']

        # ============================================================
        # Part 6: 均值恢复（以旬末/采样点为 anchor，按“日历天”skip）
        # ============================================================
        # MR(L, S) 语义（你最终需求）：
        # - lookback=L 表示窗口长度 L（按日历天）
        # - skip=S 表示从 anchor 往前跳过最近 S 个“日历天”
        # 收益区间为 [t-(L+S), t-S]
        # 例如 (40, 16)：ret = P(t-16d) / P(t-56d) - 1
        #
        # NOTE:
        # - 当目标日期不在 index 中时，取 <=目标日期 的最近可用日期（避免未来函数）
        # - 这样可兼容缺失日/不同币种缺口（价格在上游已 ffill，但索引仍按日期）
        # 中文: 参数网格规范化（L用20的倍数；S用10/20），减少“奇怪窗口”导致的噪声
        # L = [20,40,60,80,100,120], S = [10,20]
        MR_CONFIGS = [(L, S) for L in [20, 40, 60, 80, 100, 120] for S in [10, 20]]

        prices_ffill = prices.ffill()
        idx = prices_ffill.index

        def _nearest_leq(ts: pd.Timestamp) -> Optional[pd.Timestamp]:
            ts = pd.to_datetime(ts)
            pos = idx.searchsorted(ts, side='right') - 1
            if pos < 0:
                return None
            return idx[pos]

        anchor = pd.to_datetime(end_date)

        for L, S in MR_CONFIGS:
            start_target = anchor - pd.Timedelta(days=(L + S))
            end_target = anchor - pd.Timedelta(days=S)

            d_start = _nearest_leq(start_target)
            d_end = _nearest_leq(end_target)

            if d_start is None or d_end is None:
                continue
            if d_start >= d_end:
                continue

            px_start = prices_ffill.loc[d_start]
            px_end = prices_ffill.loc[d_end]

            ret = (px_end / (px_start + eps) - 1).replace([np.inf, -np.inf], 0).fillna(0)
            mr = -ret

            # MR family（按你要求：只保留 zscore + rank）
            features[f'mr_rank_{L}_skip{S}'] = mr.rank(pct=True)
            z = (mr - mr.mean()) / (mr.std() + eps)
            features[f'mr_z_{L}_skip{S}'] = z.replace([np.inf, -np.inf], 0).fillna(0)

            # 旧版本（先留注释，方便对照）
            # features[f'mr_{L}_skip{S}'] = mr
            # features[f'mr_voladj_{L}_skip{S}'] = (mr / (vol_win + eps))

        # ============================================================
        # Part 7: 旧均值恢复信号 (Mean Reversion) - 保留原有(20/60)
        # ============================================================
        for lookback in [20, 60]:
            if len(prices) >= lookback:
                mean_price = prices.iloc[-lookback:].mean()
                std_price = prices.iloc[-lookback:].std()
                z_score = (price_now - mean_price) / (std_price + eps)
                features[f'mean_reversion_{lookback}'] = (-z_score).replace([np.inf, -np.inf], 0).fillna(0)

        # ============================================================
        # Part 7.5: Volume Spike (3x or 2 sigma) & Price Appreciation (5%+)
        # ============================================================
        # 老板口径：成交量放大（3x 或 2std 异常） + 价格上涨（>=5%）
        # - 这里按“当日成交量 vs 过去窗口”做 ratio / z-score
        # - 价格上涨用过去 h 天收益（非未来）
        # NOTE: PriceAppreciation 不应被 volume 可用性绑定；否则你筛币后 volume 变稀疏会导致 price_ret 直接不生成（进而单因子 matched 0 columns）
        RET_WINDOWS = [3, 5, 10, 20]

        # ---- price appreciation (price-only) ----
        for h in RET_WINDOWS:
            if len(prices_ffill) > h:
                ret_h = (prices_ffill.iloc[-1] / (prices_ffill.iloc[-1 - h] + eps) - 1)
                ret_h = _clean(ret_h)
                features[f'price_ret_{h}d'] = ret_h
                features[f'price_app_5pct_{h}d'] = (ret_h >= 0.05).astype(float)

        # ---- Momentum family (pure price momentum) ----
        # 中文: 只做连续动量，不掺事件/成交量；用于单独的 Momentum_family
        MOM_WINDOWS = [5, 10, 20, 30]
        for h in MOM_WINDOWS:
            if len(prices_ffill) > h:
                mom_h = (prices_ffill.iloc[-1] / (prices_ffill.iloc[-1 - h] + eps) - 1)
                features[f'mom_ret_{h}d'] = _clean(mom_h)

        if getattr(self, 'df_volume', None) is not None:
            # 优先用前面“只填内部缺口”的 vol_df_local（如果有）
            if 'vol_df_local' in locals() and vol_df_local is not None:
                vol_df = vol_df_local.reindex(prices.index)
            else:
                vol_df = self.df_volume.loc[prices.index, assets]

            if vol_df is not None and len(vol_df) >= 5:
                vol_df = vol_df.replace([np.inf, -np.inf], 0).fillna(0)
                vol_now = vol_df.iloc[-1]

                #VOL_MA_WINDOWS = [5, 7, 10, 15, 20, 30, 50]
                #VOL_Z_WINDOWS = [10, 20, 30, 50]
                
                VOL_MA_WINDOWS = [7, 14, 21]
                VOL_Z_WINDOWS = [10,20, 30, 50] 

                # ---- volume ratio / event ----
                for w in VOL_MA_WINDOWS:
                    if len(vol_df) >= w:
                        v_ma = vol_df.iloc[-w:].mean()
                        v_ratio = vol_now / (v_ma + eps)
                        v_ratio = _clean(v_ratio)
                        features[f'vol_ratio_{w}'] = v_ratio
                        # features[f'vol_spike_3x_{w}'] = (v_ratio >= 3.0).astype(float)

                # ---- volume zscore / event ----
                for w in VOL_Z_WINDOWS:
                    if len(vol_df) >= w:
                        mu = vol_df.iloc[-w:].mean()
                        sd = vol_df.iloc[-w:].std()
                        v_z = (vol_now - mu) / (sd + eps)
                        v_z = _clean(v_z)
                        # features[f'vol_z_{w}'] = v_z
                        # features[f'vol_spike_2sigma_{w}'] = (v_z >= 2.0).astype(float)

                                # ---- combined events (volume spike AND price +5%) ----
                for w in VOL_MA_WINDOWS:
                    k_ratio = f'vol_ratio_{w}'
                    if k_ratio in features:
                        v_ratio = features[k_ratio]
                        for h in RET_WINDOWS:
                            k_ret = f'price_ret_{h}d'
                            if k_ret in features:
                                ret_h = features[k_ret]
                                features[f'vol3x_and_price5_{w}_{h}d'] = ((v_ratio >= 3.0) & (ret_h >= 0.05)).astype(float)

                for w in VOL_Z_WINDOWS:
                    k_z = f'vol_z_{w}'
                    if k_z in features:
                        v_z = features[k_z]
                        for h in RET_WINDOWS:
                            k_ret = f'price_ret_{h}d'
                            if k_ret in features:
                                ret_h = features[k_ret]
                                # features[f'vol2sigma_and_price5_{w}_{h}d'] = ((v_z >= 2.0) & (ret_h >= 0.05)).astype(float)

        # ============================================================
        # Part 7.6: Z-score vs 50 Day MA
        # ============================================================
        # 这里用：dev = P/MA50 - 1，然后对 dev 做滚动 z-score
        if len(prices_ffill) >= 55:
            ma50 = prices_ffill.rolling(50).mean()
            ma50_now = ma50.iloc[-1]
            dev50_now = (prices_ffill.iloc[-1] / (ma50_now + eps) - 1)
            dev50_now = _clean(dev50_now)
            features['ma50_dev'] = dev50_now

            dev_ts = (prices_ffill / (ma50 + eps) - 1)
            DEV_Z_WINDOWS = [20, 40, 80, 120]
            for w in DEV_Z_WINDOWS:
                if len(dev_ts) >= w:
                    mu = dev_ts.iloc[-w:].mean()
                    sd = dev_ts.iloc[-w:].std()
                    z = (dev50_now - mu) / (sd + eps)
                    z = _clean(z)
                    features[f'ma50_dev_z_{w}'] = z
                    features[f'ma50_dev_z_gt2sigma_{w}'] = (z >= 2.0).astype(float)

            # 简单穿越事件（只用 t-1 -> t）
            if len(dev_ts) >= 2:
                dev_prev = dev_ts.iloc[-2].replace([np.inf, -np.inf], 0).fillna(0)
                features['ma50_cross_up'] = ((dev_prev <= 0) & (dev50_now > 0)).astype(float)
                features['ma50_cross_dn'] = ((dev_prev >= 0) & (dev50_now < 0)).astype(float)

            # MA50 斜率/趋势
            for h in [5, 10, 20]:
                if len(ma50) > h:
                    slope = (ma50_now / (ma50.iloc[-1 - h] + eps) - 1)
                    features[f'ma50_slope_{h}d'] = _clean(slope)

        # ============================================================
        # Part 8: 其他衍生特征
        # ============================================================
        # 换手率估计 (如果有volume数据)
        
        # 转为DataFrame
        df_features = pd.DataFrame(features)
        # 统一清理：避免 NaN/inf 在后续统计/画图中扩散
        df_features = df_features.replace([np.inf, -np.inf], 0).fillna(0)

        df_features['date'] = end_date
        df_features['asset'] = df_features.index

        return df_features
    
    def _nearest_available_date(self, date: pd.Timestamp) -> Optional[pd.Timestamp]:
        """返回 <=date 的最近可用交易日 / Nearest available trading day <= date"""
        avail = self.df_prices.index[self.df_prices.index <= pd.to_datetime(date)]
        if len(avail) == 0:
            return None
        return avail[-1]

    def _next_month_end(self, date: pd.Timestamp, forward_months: int = 1) -> Optional[pd.Timestamp]:
        """返回 date 之后第 forward_months 个“月末交易日” / Next month-end trading day"""
        if self._month_end_dates is None or len(self._month_end_dates) == 0:
            return None

        d = pd.to_datetime(date)
        future = self._month_end_dates[self._month_end_dates > d]
        if len(future) < forward_months:
            return None
        return future[forward_months - 1]

    def compute_labels(self, date, forward_days=20, universe_size=100,
                       label_mode: str = 'month_end', forward_months: int = 1,
                       forward_decads: int = 1):
        """
        计算标签 / Build label

        默认：月度再平衡label（绝对收益）
        Default: month-end absolute return label (aligned with monthly rebalance)

        label_mode:
          - 'month_end':  label = P(next_month_end)/P(date) - 1
          - 'decad_end':  label = P(next_decad_end)/P(date) - 1 (旬度)
          - 'days':       label = P(t+forward_days)/P(t) - 1 (legacy)
        """
        date = pd.to_datetime(date)
        last_date = self._nearest_available_date(date)
        if last_date is None:
            return None

        if label_mode == 'days':
            future_dates = self.df_prices.index[self.df_prices.index > last_date]
            if len(future_dates) < forward_days:
                return None
            future_date = future_dates[forward_days - 1]
        elif label_mode == 'decad_end':
            # 旬度label：下一个旬末的收益
            future_date = self._next_decad_end(last_date, forward_decads=forward_decads)
            if future_date is None:
                return None
        else:
            # 默认月度
            future_date = self._next_month_end(last_date, forward_months=forward_months)
            if future_date is None:
                return None

        # 价格：用收盘价 close (你这里df_prices就是close)
        # Price: close-to-close return
        curr_price = self.df_prices.loc[last_date]
        future_price = self.df_prices.loc[future_date]

        fwd_ret = future_price / curr_price - 1

        mcaps = self.df_mcap.loc[last_date].dropna()
        if mcaps.empty:
            return None

        # 候选池口径：见 compute_features()
        # 先在 Excel 池子里取，再做动态 mcap TopK 过滤
        try:
            if isinstance(getattr(self, 'base_universe', None), list) and len(self.base_universe) > 0:
                _bu = [x for x in self.base_universe if x in mcaps.index]
                if len(_bu) > 0:
                    mcaps = mcaps.reindex(_bu).dropna()
        except Exception:
            pass

        try:
            _topk = getattr(self, 'mcap_topk_filter', None)
            if _topk is not None:
                _topk = int(_topk)
                if _topk > 0 and _topk < len(mcaps):
                    mcaps = mcaps.nlargest(_topk)
        except Exception:
            pass

        try:
            _u = None if universe_size is None else int(universe_size)
        except Exception:
            _u = None

        if mcaps.empty:
            return None

        if _u is not None and _u > 0 and _u < len(mcaps):
            mcaps = mcaps.nlargest(_u)

        assets = mcaps.index

        # ===== label: safe internal ffill (no bfill) =====
        # 中文: 很多币在 future_date 当天会缺价；如果直接算 future/curr，会导致 label 掉光。
        #       这里仅在 [first_valid, last_valid] 区间内 ffill，避免用未来数据。
        px_raw = self.df_prices.loc[:future_date, assets].dropna(how='all')
        if px_raw is None or len(px_raw) == 0:
            return None

        def _ffill_between_valid_df_local(df: pd.DataFrame):
            out = df.copy()
            for c in list(out.columns):
                s = out[c]
                fv = s.first_valid_index()
                lv = s.last_valid_index()
                if fv is None or lv is None:
                    continue
                out.loc[fv:lv, c] = s.loc[fv:lv].ffill()
            return out

        px = _ffill_between_valid_df_local(px_raw)

        # asof fallback（极少数情况下 last_date/future_date 不在 index）
        if last_date not in px.index:
            _ld = px.index[px.index <= last_date]
            if len(_ld) == 0:
                return None
            last_date = _ld.max()

        if future_date not in px.index:
            _fd = px.index[px.index <= future_date]
            if len(_fd) == 0:
                return None
            future_date = _fd.max()

        curr_price = px.loc[last_date]
        future_price = px.loc[future_date]
        fwd_ret = (future_price / curr_price - 1).replace([np.inf, -np.inf], np.nan)

        return fwd_ret.dropna()